# 1. Problem Definition & Scoping
**Objective:** Build a predictive machine learning model to estimate the median house value (`MedHouseVal`) in California based on neighborhood demographics.
**Success Criteria:** Achieve an $R^2$ score higher than the baseline 0.576 and minimize Mean Absolute Error (MAE).

In [ ]:
# 2. Data Collection
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

print("Fetching data from scikit-learn API...")
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("MedHouseVal")], axis=1)

display(df.head(3))

In [ ]:
# 2. Data Collection
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

print("Fetching data from scikit-learn API...")
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("MedHouseVal")], axis=1)

display(df.head(3))

In [ ]:
# 3. Data Cleaning & Preprocessing
print("Checking for missing values:")
print(df.isnull().sum())

# The California dataset has an artificial cap where all houses >$500k are listed as exactly 5.0.
# Removing these extreme, artificial outliers helps the model learn better.
df_clean = df[df['MedHouseVal'] < 5.0].copy()

print(f"\nRemaining rows after cleaning outliers: {df_clean.shape[0]}")

In [ ]:
# 4. Exploratory Data Analysis (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Plot correlation heatmap to uncover relationships
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Feature Correlation with House Value")
plt.show()

In [ ]:
# 5. Feature Engineering & Selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create smarter features based on EDA
df_clean['Rooms_per_Household'] = df_clean['AveRooms'] / df_clean['AveOccup']
df_clean['Bedrooms_per_Room'] = df_clean['AveBedrms'] / df_clean['AveRooms']
df_clean['Log_MedIncome'] = np.log1p(df_clean['MedInc']) # Normalize skewed income

# Select final features by dropping the raw ones we replaced
X = df_clean.drop(columns=['MedHouseVal', 'MedInc', 'AveRooms', 'AveBedrms'])
y = df_clean['MedHouseVal']

# Split data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Scale features so variables like population don't overpower smaller decimals
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Transform only to prevent data leakage

print("Features engineered, selected, and scaled.")

In [ ]:
# 6. Model Training & Evaluation
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train a penalized linear model (Ridge) to reduce overfitting on our new features
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# Evaluate on unseen testing data
y_pred = model.predict(X_test_scaled)

print("--- FINAL MODEL EVALUATION ---")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
print(f"R²:   {r2_score(y_test, y_pred):.3f}")

In [ ]:
# 7. Deployment & Monitoring
import joblib
from google.colab import files

# Export the trained model and scaler for deployment in a web API / Streamlit App
joblib.dump(model, 'ridge_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

print("Preparing files for production deployment...")

# Download automatically to your local machine
files.download('ridge_model.pkl')
files.download('feature_scaler.pkl')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="deep")
fig = plt.figure(figsize=(20, 16))

ax1 = plt.subplot(2, 2, 1)
sns.histplot(df['MedHouseVal'], bins=60, kde=True, color='rebeccapurple', alpha=0.7, ax=ax1)
ax1.axvline(df['MedHouseVal'].median(), color='red', linestyle='--', lw=2, label=f"Median: ${df['MedHouseVal'].median()*100000:,.0f}")
ax1.set_title("1. Distribution of House Prices", fontsize=15, weight='bold')
ax1.set_xlabel("Median House Value (in $100,000s)", fontsize=12)
ax1.set_ylabel("Frequency", fontsize=12)
ax1.legend()

ax2 = plt.subplot(2, 2, 2)
feature_names = X.columns
coefficients = model.coef_
indices = np.argsort(np.abs(coefficients))
colors = ['crimson' if c < 0 else 'seagreen' for c in coefficients[indices]]
ax2.barh(range(len(indices)), coefficients[indices], color=colors, alpha=0.85)
ax2.set_yticks(range(len(indices)))
ax2.set_yticklabels(feature_names[indices], fontsize=12)
ax2.set_title("2. Model Feature Importance", fontsize=15, weight='bold')
ax2.set_xlabel("Coefficient Weight", fontsize=12)

ax3 = plt.subplot(2, 2, 3)
hb = ax3.hexbin(y_test, y_pred, gridsize=40, cmap='YlGnBu', mincnt=1)
ax3.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3)
cb = fig.colorbar(hb, ax=ax3)
cb.set_label('Density of Houses', fontsize=12)
ax3.set_title("3. Actual vs Predicted (Density Hexbin Map)", fontsize=15, weight='bold')
ax3.set_xlabel("Actual Values", fontsize=12)
ax3.set_ylabel("Predicted Values", fontsize=12)

ax4 = plt.subplot(2, 2, 4)
sns.scatterplot(x='Log_MedIncome', y='MedHouseVal', data=df_clean, alpha=0.2, color='darkorange', edgecolor=None, ax=ax4)
sns.regplot(x='Log_MedIncome', y='MedHouseVal', data=df_clean, scatter=False, color='darkred', line_kws={"linewidth":3}, ax=ax4)
ax4.set_title("4. Income vs Price Trend", fontsize=15, weight='bold')
ax4.set_xlabel("Log of Median Income", fontsize=12)
ax4.set_ylabel("House Value", fontsize=12)

plt.tight_layout(pad=3.0)
plt.show()